# Explainability — SHAP + Attention Visualization
Interpreting XLM-R predictions on Swahili sentiment data.

In [ ]:
# ==========================================
# INSTALL
# ==========================================
!pip install shap transformers torch pandas matplotlib seaborn

In [ ]:
# ==========================================
# IMPORTS
# ==========================================
import shap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

os.makedirs('../results/graphs', exist_ok=True)
os.makedirs('../results/reports', exist_ok=True)

label_names = ['Negative', 'Neutral', 'Positive']

In [ ]:
# ==========================================
# LOAD CLEANED DATA
# ==========================================
df = pd.read_csv('../data/processed/cleaned_data.csv')
df = df.dropna(subset=['clean_text', 'label_encoded'])
df['clean_text'] = df['clean_text'].astype(str)
df['label_encoded'] = df['label_encoded'].astype(int)

_, X_test, _, y_test = train_test_split(
    df['clean_text'], df['label_encoded'],
    test_size=0.2, random_state=42,
    stratify=df['label_encoded']
)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f'Test samples: {len(X_test)}')

In [ ]:
# ==========================================
# LOAD SAVED XLM-R MODEL
# Run this AFTER transformers.ipynb is done.
# ==========================================
MODEL_PATH = '../models/transformers/xlm_roberta'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

print('Model loaded.')

In [ ]:
# ==========================================
# SHAP EXPLAINABILITY
# Uses a pipeline + SHAP Text Explainer
# ==========================================

# Build a Hugging Face pipeline
clf_pipeline = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    top_k=None,          # Return scores for all labels
    device=-1            # CPU
)

# Wrap for SHAP
def pipeline_wrapper(texts):
    results = clf_pipeline(list(texts))
    # Shape: (n_samples, n_labels)
    output = []
    for result in results:
        scores = sorted(result, key=lambda x: x['label'])
        output.append([s['score'] for s in scores])
    return np.array(output)


# Use a small background set (SHAP runs slow on CPU)
background = X_test[:20].tolist()
explainer = shap.Explainer(pipeline_wrapper, masker=shap.maskers.Text(tokenizer))

# Explain 5 test samples
sample_texts = X_test[:5].tolist()
shap_values = explainer(sample_texts)

print('SHAP values computed.')

In [ ]:
# ==========================================
# SHAP TEXT PLOT — Positive class
# ==========================================
print('SHAP Explanation for sample 0 (Positive class):')
shap.plots.text(shap_values[0, :, 2])  # index 2 = Positive

In [ ]:
# ==========================================
# SHAP BAR PLOT — Feature importance
# ==========================================
shap.plots.bar(shap_values[:, :, 2].mean(0), max_display=15,
               show=False)
plt.title('Top Features — Positive Sentiment (SHAP)')
plt.tight_layout()
plt.savefig('../results/graphs/shap_bar_positive.png', bbox_inches='tight')
plt.show()
print('SHAP bar plot saved.')

In [ ]:
# ==========================================
# ATTENTION VISUALIZATION
# Shows which tokens the model attends to.
# ==========================================
def get_attention_weights(text):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # Average across all heads and layers
    attentions = outputs.attentions  # Tuple of (batch, heads, seq, seq)
    avg_attention = torch.stack(attentions).mean(dim=[0, 1])  # (seq, seq)

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    cls_attention = avg_attention[0].numpy()  # Attention from [CLS]

    return tokens, cls_attention


def plot_attention(text, title, save_path):
    tokens, attn = get_attention_weights(text)

    # Trim padding
    attn = attn[:len(tokens)]

    plt.figure(figsize=(max(10, len(tokens) * 0.5), 3))
    plt.bar(range(len(tokens)), attn, color='steelblue')
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right', fontsize=9)
    plt.ylabel('Attention Weight')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()


# Plot for one positive and one negative example
pos_sample = X_test[y_test == 2].iloc[0]
neg_sample = X_test[y_test == 0].iloc[0]

plot_attention(
    pos_sample,
    'Attention Weights — Positive Tweet',
    '../results/graphs/attention_positive.png'
)

plot_attention(
    neg_sample,
    'Attention Weights — Negative Tweet',
    '../results/graphs/attention_negative.png'
)

In [ ]:
# ==========================================
# ERROR ANALYSIS
# Finds misclassified samples and patterns.
# ==========================================

# Get predictions on full test set
def predict_batch(texts, batch_size=16):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            list(batch), truncation=True,
            padding=True, max_length=128,
            return_tensors='pt'
        )
        with torch.no_grad():
            logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=1).numpy()
        all_preds.extend(preds)
    return np.array(all_preds)


y_pred = predict_batch(X_test)

# Build error dataframe
error_df = pd.DataFrame({
    'text':      X_test,
    'true':      y_test,
    'predicted': y_pred
})

error_df['true_label']  = error_df['true'].map({0:'Negative',1:'Neutral',2:'Positive'})
error_df['pred_label']  = error_df['predicted'].map({0:'Negative',1:'Neutral',2:'Positive'})
error_df['is_error']    = error_df['true'] != error_df['predicted']

errors = error_df[error_df['is_error']]

print(f'Total errors: {len(errors)} out of {len(error_df)} ({100*len(errors)/len(error_df):.1f}%)')
print('\nError breakdown by true label:')
print(errors['true_label'].value_counts())

In [ ]:
# ==========================================
# ERROR HEATMAP — Confusion by class
# ==========================================
error_pivot = errors.groupby(['true_label', 'pred_label']).size().unstack(fill_value=0)

plt.figure(figsize=(6, 4))
sns.heatmap(error_pivot, annot=True, fmt='d', cmap='Reds')
plt.title('Error Pattern Heatmap (True vs Predicted)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../results/graphs/error_heatmap.png')
plt.show()

In [ ]:
# ==========================================
# SAMPLE ERROR CASES
# ==========================================
print('=== Sample Misclassified Tweets ===\n')

for _, row in errors.sample(min(10, len(errors)), random_state=42).iterrows():
    print(f'Text:      {row["text"][:100]}')
    print(f'True:      {row["true_label"]}')
    print(f'Predicted: {row["pred_label"]}')
    print('-' * 60)

In [ ]:
# ==========================================
# SAVE ERROR REPORT
# ==========================================
errors.to_csv('../results/reports/error_analysis.csv', index=False)
print('Error analysis saved to ../results/reports/error_analysis.csv')

# Full classification report
report = classification_report(y_test, y_pred, target_names=label_names)
print('\n=== Final Classification Report (XLM-R) ===')
print(report)

with open('../results/reports/xlmr_classification_report.txt', 'w') as f:
    f.write(report)
print('Classification report saved.')